# COMPLETE MODEL

In [ ]:
import pandas as pd
import numpy as np
import os
import cv2
import tensorflow as tf
from tensorflow import keras
from keras.applications.vgg16 import VGG16
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path_di = "/content/drive/MyDrive/brain_tumor_dataset"
data = []
labels = []

for category in ["yes", "no"]:
    folder = os.path.join(path_di, category)
    label = 1 if category == "yes" else 0 #label encoding

    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (128, 128)) #resizing the pixles

        img = img / 255.0   # FIX: normalize input

        data.append(img)
        labels.append(label)

X = np.array(data)
y = np.array(labels)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
base_model = VGG16(
    include_top=False,            # FIX
    weights="imagenet",
    input_shape=(128, 128, 3)
)

In [ ]:
base_model.trainable = False

In [ ]:
model = keras.Sequential([
    base_model,
    keras.layers.Flatten(), #converts 3D vectors int 1D vector
    keras.layers.Dense(128, activation="relu"), #fully connected layer
    keras.layers.Dropout(0.3), #30% of the inputs to the next layer, purpose to preventing from overfitting
    keras.layers.Dense(1, activation="sigmoid")
])


# Results:
# 7/7 ━━━━━━━━━━━━━━━━━━━━ 58s 8s/step - accuracy: 0.9554 - loss: 0.1525 - val_accuracy: 0.8235 - val_loss: 0.4383


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 674s 5s/step - accuracy: 0.9323 - loss: 0.1665 - val_accuracy: 0.9440 - val_loss: 0.1445
Epoch 2/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 698s 5s/step - accuracy: 0.9515 - loss: 0.1325 - val_accuracy: 0.9523 - val_loss: 0.1289
Epoch 3/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 674s 5s/step - accuracy: 0.9557 - loss: 0.1108 - val_accuracy: 0.9367 - val_loss: 0.1628
Epoch 4/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 680s 5s/step - accuracy: 0.9665 - loss: 0.0916 - val_accuracy: 0.9651 - val_loss: 0.0951
Epoch 5/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 683s 5s/step - accuracy: 0.9696 - loss: 0.0827 - val_accuracy: 0.9596 - val_loss: 0.1089
Epoch 6/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 677s 5s/step - accuracy: 0.9776 - loss: 0.0658 - val_accuracy: 0.9697 - val_loss: 0.0940
Epoch 7/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 679s 5s/step - accuracy: 0.9750 - loss: 0.0650 - val_accuracy: 0.9587 - val_loss: 0.1079
Epoch 8/20
137/137 ━━━━━━━━━━━━━━━━━━━━ 671s 5s/step - accuracy: 0.9770 - loss: 0.0685 - val_accu

In [ ]:
from keras.applications.vgg16 import preprocess_input


def brain_tumor(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128, 128))
    img = preprocess_input(img).reshape(1, 128, 128, 3)

    pred = model.predict(img)[0][0]
    return "Tumor" if pred > 0.5 else "No Tumor", float(pred)


In [ ]:
result = brain_tumor("/content/drive/MyDrive/brain_tumor_dataset/yes/Yes (101).jpeg")
print(result)

# ('No Tumor', 3.1316820248150634e-09)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
('Tumor', 1.0)


In [ ]:
model.save("brain_tumor.keras")

In [ ]:
!cp brain_tumor.keras /content/drive/MyDrive/

In [ ]:
from google.colab import files
files.download("brain_tumor.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Model Performance

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "/content/drive/MyDrive/brain_tumor.keras",
    compile=False
)


In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_PATH = "/content/drive/MyDrive/brain_tumor_dataset"

data = []
labels = []

for category in ["yes", "no"]:
    folder = os.path.join(DATASET_PATH, category)
    label = 1 if category == "yes" else 0

    for file in os.listdir(folder):
        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (128, 128))
        img = img / 255.0   # SAME AS TRAINING
        data.append(img)
        labels.append(label)

X = np.array(data)
y = np.array(labels)

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
y_prob = model.predict(X_test, batch_size=32)
y_pred = (y_prob > 0.5).astype(int)


35/35 ━━━━━━━━━━━━━━━━━━━━ 215s 6s/step


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Accuracy : 0.9862385321100917
Precision: 0.9804270462633452
Recall   : 0.9927927927927928
F1-score : 0.9865711727842436

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.99       535
           1       0.98      0.99      0.99       555

    accuracy                           0.99      1090
   macro avg       0.99      0.99      0.99      1090
weighted avg       0.99      0.99      0.99      1090


Confusion Matrix:

[[524  11]
 [  4 551]]
